# 갑상선암 검진 정책 변화 분석 (GitHub + Colab)

이 노트북은 GitHub 저장소에 올려둔 CSV를 **raw URL로 직접 읽어**, Colab에서 재현 가능하게 분석하도록 구성했습니다.

핵심 출력:
- 전체/남녀 연령표준화발생률 추세
- 2012=100 충격지수
- 버터플라이 3단 비교
- 연령 heatmap
- 감소율 / rebound 워터폴
- 버블차트
- ITS 분석


In [ ]:
!apt-get -qq install fonts-nanum > /dev/null
!pip -q install pandas numpy matplotlib statsmodels openpyxl scipy

In [ ]:
from pathlib import Path

GITHUB_USER = "your-github-id"
REPO_NAME = "your-repo-name"
BRANCH = "main"
DATA_DIR = "data"

MAIN_FILE = "국립암센터_24개종 암발생률_20260120.csv"
AGE_FILE = "24개_암종_성_연령_5세_별_암발생자수__발생률_20260324142549.csv"

REPO_RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{DATA_DIR}"
OUTDIR = "thyroid_analysis_outputs"
print("REPO_RAW_BASE =", REPO_RAW_BASE)

In [ ]:
import sys, os, urllib.request

# GitHub에서 노트북을 직접 열었을 때 분석 스크립트도 raw로 불러옵니다.
script_candidates = [
    f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/src/thyroid_colab_analysis.py",
    f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/thyroid_colab_analysis.py",
]
for url in script_candidates:
    try:
        urllib.request.urlretrieve(url, "/content/thyroid_colab_analysis.py")
        print("분석 스크립트 다운로드 성공:", url)
        break
    except Exception:
        continue

sys.path.append("/content")
from thyroid_colab_analysis import run_analysis, infer_repo_raw

In [ ]:
from google.colab import files

main_source = infer_repo_raw(REPO_RAW_BASE, MAIN_FILE)
age_source = infer_repo_raw(REPO_RAW_BASE, AGE_FILE)

def url_exists(url: str) -> bool:
    import urllib.request
    try:
        with urllib.request.urlopen(url) as r:
            return r.status == 200
    except Exception:
        return False

if not url_exists(main_source) or not url_exists(age_source):
    print("GitHub raw 데이터 접근에 실패했습니다. 업로드로 전환합니다.")
    uploaded = files.upload()
    main_source = f"/content/{MAIN_FILE}" if MAIN_FILE in uploaded else None
    age_source = f"/content/{AGE_FILE}" if AGE_FILE in uploaded else None

if not main_source or not age_source:
    raise FileNotFoundError("CSV 2개를 찾지 못했습니다. GitHub raw 경로 또는 업로드 파일명을 확인하세요.")

print("main_source =", main_source)
print("age_source =", age_source)

In [ ]:
results = run_analysis(main_source=main_source, age_source=age_source, outdir=OUTDIR)
print("완료. 결과 폴더:", results["outdir"])

In [ ]:
display(results["summary_change"])
display(results["age_change_total"].sort_values("2012→2015 변화율(%)").head(10))
display(results["age_change_total"].sort_values("2015→2023 변화율(%)", ascending=False).head(10))

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/thyroid_analysis_outputs", "zip", OUTDIR)
files.download(zip_path)